In [ ]:
# Set knit directory
setwd(here::here('./07_euc_sideroxylum'))
knitr::opts_knit$set(root.dir = setwd(here::here('./07_euc_sideroxylum')))
getwd()

In [ ]:
options(scipen = 999)

# Save figures in specific place
knitr::opts_chunk$set(autodep        = TRUE,
                      cache          = FALSE,
                      cache.comments = TRUE,
                      comment = "",
                      # Save figures as pdf ?
                      #dev = c( "png", "pdf"),
                  
                      # Include code?
                      echo           = TRUE,
                      error          = FALSE,
                      fig.align      = "center",
                      
                      # Path where figures are going to be store pdf single 
                      # figures
                      #fig.path       = paste0("./figures_euc_sideroxylum", "/"),
                      fig.width      = 15,
                      fig.height     = 13,
                      message        = FALSE,
                      warning        = FALSE)

In [ ]:
library(HIEdroughtbox)
library(ggplot2)

# For data manipulation
library(dplyr)
library(purrr)
library(stringr)
library(tidyr)
library(here)

# For adding regression line to ggplot
library(ggpubr)

In [ ]:

tree_id_colors <- c("131" = "#eec000",
                   "132" = "#cf544c",
                   "133" = "#0175c3",
                   "134" = "#878687",
                   
                   "135" =  "#000000",
                   "136" =  "#008b28",
                   "137" =  "#8600b6",
                   "138" =  "#00a0ab")

string_colors <- c(string_1 = "#eec000",
                   string_2 = "#cf544c",
                   string_3 = "#0175c3",
                   string_4 = "#878687",
                   
                   string_5 =  "#000000",
                   string_6 =  "#008b28",
                   string_7 =  "#8600b6",
                   string_8 =  "#00a0ab")


# Function for adding the number of observations on top of a boxplot
give_n <- function(x){
  return(c(y = median(x)*2, label = length(x))) 
  # experiment with the multiplier to find the perfect position
}

# 1. Load data

## Load water loss data

In [ ]:
load("./euc_sideroxylum_processed_data/euc_sideroxylum_water_loss_data.RData")

In [ ]:
# Show the names of each element in the list
glimpse(euc_sideroxylum_water_loss_data)

## Load cleaned droughtbox data

In [ ]:
load("./euc_sideroxylum_processed_data/euc_sideroxylum_rwc_filtered_data.RData")

euc_sideroxylum_data_for_gres <- euc_sideroxylum_data_for_water_loss

In [ ]:
# Show the names of each element in the list
glimpse(euc_sideroxylum_data_for_gres)

# 2. Calculate species residual conductance 

## Calculate Residual conductance

In [ ]:
# Atmospheric pressure in the droughtbox constant -------------------------------
atmospheric_pressure_constant = 101.6

# Get VPD data ------------------------------------------------------------------

euc_sideroxylum_vpd_data <-
        euc_sideroxylum_data_for_gres %>%
        dplyr::select(vpd_control,
                      time_seconds,
                      string_number,
                      set_temperature,
                      vpd_avg_kpa_avg,
                      leaf_dry_weight_g,
                      sample_dry_weight_g,
                      branch_dry_matter_content_g_g 
                      )

In [ ]:
# Filter transpiration rates ----------------------------------------------------

euc_sideroxylum_water_loss_data <-
  
        euc_sideroxylum_water_loss_data %>% 
        dplyr::select(spcode,
                      tree_id,
                      areas_m2,
                      vpd_control,
                      leaf_areas_m2, 
                      string_number,
                      set_temperature,
                      transpiration_double_grams_per_sec_m2) %>% 
        select(spcode, tree_id, string_number,set_temperature, vpd_control, everything())


In [ ]:
# Estimate residual conductance -------------------------------------------------

euc_sideroxylum_residual_conductance_df <-

        # Join data
        dplyr::inner_join(euc_sideroxylum_water_loss_data, 
                         euc_sideroxylum_vpd_data,
                         by = join_by(string_number, 
                                      set_temperature, 
                                      vpd_control)) %>% 

        # Sort columns
        dplyr::select(spcode, time_seconds, everything()) %>%

        # Calculate variables
        dplyr::mutate(

            # Transform grams of water to mols of water
            transpiration_double_mol_per_sec_m2 = transpiration_double_grams_per_sec_m2/18.015,

            # Transform transpiration to Milimols of water
            transpiration_double_mmol_per_sec_m2 = (transpiration_double_grams_per_sec_m2/18.015)*1000,

            # Calculate gres as (E/VPD)*101.6
            gres = (transpiration_double_mol_per_sec_m2/vpd_avg_kpa_avg)*101.6) %>% 


        dplyr::group_by(tree_id,
                        areas_m2,
                        vpd_control, 
                        leaf_areas_m2, 
                        set_temperature, 
                        leaf_dry_weight_g,
                        sample_dry_weight_g,
                        branch_dry_matter_content_g_g
                        ) %>% 

        # Get the median values
        dplyr::summarise(gres_mols_m2_s = median(gres),
                         transpiration_double_H2Ommol_per_sec_m2 = median(transpiration_double_mmol_per_sec_m2)) %>% 
        
        select(tree_id, 
               vpd_control, 
               set_temperature, 
               sample_dry_weight_g, 
               everything())


In [ ]:
euc_sideroxylum_residual_conductance_df %>%
    mutate(across(where(is.numeric), round, 3)) %>%
    reactable::reactable(.)

### Plot Residual conductance data

In [ ]:

ggplot(data = euc_sideroxylum_residual_conductance_df, aes(x = set_temperature,
                                                     y = gres_mols_m2_s,
                                                  
                             colour = factor(tree_id))) +
        
        theme_bw(base_size = 14) +
  
        geom_point(size = 3.5) +
  
        # Increase the tickmarks of the axes
        scale_y_continuous(breaks = scales::pretty_breaks(n = 12)) + 
        scale_x_continuous(breaks = scales::pretty_breaks(n = 7)) +
  
        # Position label at the bottom
        theme(legend.position = "bottom") +
    
        facet_wrap(~vpd_control) +
        
        # Set colors
        #scale_color_manual(values = c("#9e1523","#293352")) +
        ylab('Residual conductance (mols*m-2*s-1)')+
        xlab("Temperature") +
        scale_color_manual(values = tree_id_colors) 

### Plot transpiration data

In [ ]:

ggplot(data = euc_sideroxylum_residual_conductance_df, aes(x = set_temperature,
                                                     y = transpiration_double_H2Ommol_per_sec_m2,
                                                     colour = factor(tree_id))) +
        
        theme_bw(base_size = 14) +
  
        geom_point(size = 3.5) +
  
        # Increase the tickmarks of the axes
        scale_y_continuous(breaks = scales::pretty_breaks(n = 12)) + 
        scale_x_continuous(breaks = scales::pretty_breaks(n = 7)) +
  
        # Position label at the bottom
        theme(legend.position = "bottom") +
    
        facet_wrap(~vpd_control) +
        
        # Set colors
        #scale_color_manual(values = c("#9e1523","#293352")) +
        ylab('Transpiration (mmols*m-2*s-1)')+
        xlab("Temperature") +
        scale_color_manual(values = tree_id_colors) 

In [ ]:
euc_sideroxylum_residual_conductance_df <- 
  
    euc_sideroxylum_residual_conductance_df %>% 
        dplyr::mutate(sla = leaf_areas_m2/leaf_dry_weight_g,
                      branch_sla =  areas_m2/sample_dry_weight_g,
                      lma = 1/sla,
                      branch_lma = 1/branch_sla, .keep = "unused")

In [ ]:

euc_sideroxylum_residual_conductance_df %>% 
    
    ggplot(data = ., aes(x = lma, y= gres_mols_m2_s))+
    geom_point() +
    geom_smooth(method = "lm", se = TRUE) +
    facet_grid(~vpd_control)

In [ ]:

euc_sideroxylum_residual_conductance_df %>% 
    
    ggplot(data = ., aes(x = lma, y = transpiration_double_H2Ommol_per_sec_m2)) +
    geom_point() +
    geom_smooth(method = "lm", se = TRUE) +
    facet_grid(~vpd_control)

In [ ]:
# Add spcode to data before saving 

euc_sideroxylum_residual_conductance_data <- 
  
  euc_sideroxylum_residual_conductance_df %>% 
    
  mutate(spcode = "euc_sideroxylum") %>% 
  ungroup() %>% 
  select(-c(sample_dry_weight_g, areas_m2, leaf_areas_m2, leaf_dry_weight_g)) %>% 
  select(spcode, everything())

# 3. Export data

In [ ]:
save(euc_sideroxylum_residual_conductance_data, 
     file = "./euc_sideroxylum_processed_data/euc_sideroxylum_residual_conductance_data.RData")